# Entrainement et versioning du modele

Ce notebook reprend le pipeline actuel du projet `ml_conso` : chargement des donnees, creation de la cible `evo_conso`, selection des features, entrainement, evaluation et sauvegarde du modele actif.

## 1. Initialiser les chemins du projet

In [21]:
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "src" / "ml_conso").exists():
    PROJECT_ROOT = cwd
elif (cwd / "ML_CONSO_regression" / "src" / "ml_conso").exists():
    PROJECT_ROOT = cwd / "ML_CONSO_regression"
else:
    PROJECT_ROOT = cwd

SYS_PATH = PROJECT_ROOT

if str(SYS_PATH) not in sys.path:
    sys.path.insert(0, str(SYS_PATH))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Projet    :", PROJECT_ROOT)
print("Src       :", SYS_PATH)
print("Artifacts :", ARTIFACTS_DIR)

Projet    : c:\Users\chris\Desktop\ml-filrouge-project - v3\ML_CONSO_regression
Src       : c:\Users\chris\Desktop\ml-filrouge-project - v3\ML_CONSO_regression
Artifacts : c:\Users\chris\Desktop\ml-filrouge-project - v3\ML_CONSO_regression\artifacts


## 2. Importer les fonctions du projet

In [22]:
import json

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from src.ml_conso.data import load_data
from src.ml_conso.evaluate import evaluate_model
from src.ml_conso.features import FEATURES, TARGET, select_features, split_data
from src.ml_conso.pipeline import (
    activate_model_version,
    build_pipeline,
    export_model_contract,
    save_model_version,
)
from src.ml_conso.train import log_experiment

## 3. Charger les donnees

In [23]:
df_raw = load_data()

print("Shape brute :", df_raw.shape)
df_raw.head()

WARN: data file not found at C:\Users\chris\Desktop\ml-filrouge-project - v3\ML_CONSO_regression\Merge_conso_meteo_soleil_090426.csv — creating synthetic dataset for dev.
Shape brute : (200, 8)


,Date,Conso_MWH,DUREE_ENSOLEILLEMENT,MOYENNE_TEMP_HORAIRES_SA_PONDEREE,TEMP_MAX_SA,MOYENNE_HUMIDITES_RELATIVES_HORAIRES,TEMP_MIN_SOUS_ABRI,CODE_DEPARTEMENT
0,2020-01-01,3952.874786,0.884498,28.061349,27.295151,70.899915,4.412157,2
1,2020-01-02,2783.932049,5.902752,9.440817,15.071191,72.517508,17.185092,3
2,2020-01-03,3826.861149,6.734564,4.012987,19.677209,48.959599,14.378954,2
3,2020-01-04,3583.903465,0.233402,7.152016,32.103252,66.599973,-2.449088,2
4,2020-01-05,4183.383673,5.053950,1.525162,13.723811,38.280456,10.353335,3


## 4. Verifier les colonnes utiles

In [24]:
required_columns = ["Date", "Conso_MWH"] + FEATURES
missing_columns = [col for col in required_columns if col not in df_raw.columns]

print("Target :", TARGET)
print("Features :", FEATURES)
print("Colonnes manquantes :", missing_columns)

assert not missing_columns, f"Colonnes manquantes : {missing_columns}"

Target : evo_conso
Features : ['DUREE_ENSOLEILLEMENT', 'MOYENNE_TEMP_HORAIRES_SA_PONDEREE', 'TEMP_MAX_SA', 'MOYENNE_HUMIDITES_RELATIVES_HORAIRES', 'TEMP_MIN_SOUS_ABRI']
Colonnes manquantes : []


## 5. Creer la cible et selectionner les features

In [25]:
df_model = select_features(df_raw.copy())

print("Shape modele :", df_model.shape)
print("Colonnes modele :", df_model.columns.tolist())
df_model.head()

Shape modele : (200, 6)
Colonnes modele : ['DUREE_ENSOLEILLEMENT', 'MOYENNE_TEMP_HORAIRES_SA_PONDEREE', 'TEMP_MAX_SA', 'MOYENNE_HUMIDITES_RELATIVES_HORAIRES', 'TEMP_MIN_SOUS_ABRI', 'evo_conso']


,DUREE_ENSOLEILLEMENT,MOYENNE_TEMP_HORAIRES_SA_PONDEREE,TEMP_MAX_SA,MOYENNE_HUMIDITES_RELATIVES_HORAIRES,TEMP_MIN_SOUS_ABRI,evo_conso
0,0.884498,28.061349,27.295151,70.899915,4.412157,0.535749
1,5.902752,9.440817,15.071191,72.517508,17.185092,0.146759
2,6.734564,4.012987,19.677209,48.959599,14.378954,0.493816
3,0.233402,7.152016,32.103252,66.599973,-2.449088,0.352210
4,5.053950,1.525162,13.723811,38.280456,10.353335,0.392108


## 6. Controler la qualite des donnees modele

In [26]:
missing_summary = df_model.isna().sum().sort_values(ascending=False)
target_summary = df_model[TARGET].describe()

display(missing_summary.to_frame("missing_values"))
display(target_summary.to_frame("evo_conso"))

,missing_values
DUREE_ENSOLEILLEMENT,0
MOYENNE_TEMP_HORAIRES_SA_PONDEREE,0
TEMP_MAX_SA,0
MOYENNE_HUMIDITES_RELATIVES_HORAIRES,0
TEMP_MIN_SOUS_ABRI,0
evo_conso,0


,evo_conso
count,2.000000e+02
mean,9.769963e-17
std,4.546131e-01
min,-1.034160e+00
25%,-3.497340e-01
50%,2.123879e-02
75%,3.368018e-01
max,9.895457e-01


## 7. Split train / test

In [27]:
X_train, X_test, y_train, y_test = split_data(df_model)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

assert TARGET not in X_train.columns
assert y_train.name == TARGET

X_train : (160, 5)
X_test  : (40, 5)
y_train : (160,)
y_test  : (40,)


## 8. Construire le pipeline

In [28]:
pipeline = build_pipeline()

print(pipeline)
print("Etapes :", list(pipeline.named_steps))

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('sunshine',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['DUREE_ENSOLEILLEMENT']),
                                                 ('temp_log',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log',
                                                                   FunctionTransformer(func=<function log_transform at 0x000001EEEE4B05E0>)),
                        

## 9. Entrainer le modele

In [29]:
pipeline.fit(X_train, y_train)

print("Modele entraine")

Modele entraine


## 10. Evaluer le modele

In [30]:
metrics = evaluate_model(pipeline, X_test, y_test)

metrics_df = pd.DataFrame([metrics])
display(metrics_df)
metrics

,rmse,mae,r2
0,0.465157,0.367754,0.070009


{'rmse': 0.46515728192625827,
 'mae': 0.36775366500196616,
 'r2': 0.07000927216731889}

## 11. Sauvegarder et activer le modele courant

In [31]:
mlruns_dir = PROJECT_ROOT / "mlruns"
if mlruns_dir.exists():
    for exp_dir in mlruns_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        meta_file = exp_dir / "meta.yaml"
        if not meta_file.exists():
            content = (
                f"artifact_location: {exp_dir.as_uri()}\n"
                f"name: {exp_dir.name}\n"
                "lifecycle_stage: active\n"
                f"experiment_id: {exp_dir.name}\n"
            )
            try:
                meta_file.write_text(content, encoding="utf-8")
            except OSError as error:
                print(
                    f"Warning: unable to write meta.yaml for {exp_dir}: {error}"
                )

mlflow.set_experiment("electricity_forecasting")

with mlflow.start_run():
    log_experiment(metrics)

    saved_paths = save_model_version(pipeline, metrics, "current", ARTIFACTS_DIR)
    latest_paths = activate_model_version("current", ARTIFACTS_DIR)

    mlflow.sklearn.log_model(sk_model=pipeline, artifact_path="model")

    feature_columns_path = ARTIFACTS_DIR / "conso_feature_columns.pkl"
    joblib.dump(X_train.columns.tolist(), feature_columns_path)

    contract_path = export_model_contract(X_train.columns.tolist(), ARTIFACTS_DIR)

print("Version current :", saved_paths)
print("Version latest  :", latest_paths)
print("Features       :", feature_columns_path)
print("Contract       :", contract_path)

2026/05/21 22:17:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Version current : {'model': WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project - v3/ML_CONSO_regression/artifacts/models/conso_model_current.joblib'), 'metrics': WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project - v3/ML_CONSO_regression/artifacts/metrics/conso_metrics_current.json')}
Version latest  : {'model': WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project - v3/ML_CONSO_regression/artifacts/models/conso_model_latest.joblib'), 'metrics': WindowsPath('c:/Users/chris/Desktop/ml-filrouge-project - v3/ML_CONSO_regression/artifacts/metrics/conso_metrics_latest.json')}
Features       : c:\Users\chris\Desktop\ml-filrouge-project - v3\ML_CONSO_regression\artifacts\conso_feature_columns.pkl
Contract       : c:\Users\chris\Desktop\ml-filrouge-project - v3\ML_CONSO_regression\artifacts\model_contract.json


## 12. Verifier les artefacts generes

In [32]:
expected_artifacts = [
    MODELS_DIR / "conso_model_current.joblib",
    MODELS_DIR / "conso_model_latest.joblib",
    METRICS_DIR / "conso_metrics_current.json",
    METRICS_DIR / "conso_metrics_latest.json",
    ARTIFACTS_DIR / "conso_model_contract.json",
    ARTIFACTS_DIR / "conso_feature_columns.pkl",
]

artifact_status = pd.DataFrame(
    {
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in expected_artifacts],
        "exists": [path.exists() for path in expected_artifacts],
    }
)

display(artifact_status)
assert artifact_status["exists"].all()

,path,exists
0,artifacts\models\conso_model_current.joblib,True
1,artifacts\models\conso_model_latest.joblib,True
2,artifacts\metrics\conso_metrics_current.json,True
3,artifacts\metrics\conso_metrics_latest.json,True
4,artifacts\conso_model_contract.json,True
5,artifacts\conso_feature_columns.pkl,True


## 13. Lire le contrat du modele

In [33]:
with (ARTIFACTS_DIR / "conso_model_contract.json").open(encoding="utf-8") as file:
    model_contract = json.load(file)

model_contract

{'model_path': 'artifacts/models/conso_model_latest.joblib',
 'metrics_path': 'artifacts/metrics/conso_metrics_latest.json',
 'features': ['DUREE_ENSOLEILLEMENT',
  'MOYENNE_TEMP_HORAIRES_SA_PONDEREE',
  'TEMP_MAX_SA',
  'MOYENNE_HUMIDITES_RELATIVES_HORAIRES',
  'TEMP_MIN_SOUS_ABRI'],
 'target': 'evo_conso',
 'format': 'joblib',
 'usage': 'Load in the backend with joblib,\n        expose predictions through an API for the frontend.'}

## 14. Pistes d'amelioration du modele

1. Reintegrer des variables temporelles : jour de semaine, semaine de l'annee, mois, saison.
2. Reintegrer `CODE_DEPARTEMENT` pour capter les differences geographiques.
3. Ajouter davantage de variables meteo : vent, precipitation, neige, moyenne temperature, amplitude thermique.
4. Comparer le pipeline actuel a l'ancien modele enrichi `v1/v2`, qui utilisait plus de features.
5. Tester plusieurs modeles : ExtraTreesRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor.
6. Ajouter une validation croisee ou un split temporel si l'objectif est de predire des periodes futures.
7. Suivre un seuil anti-regression dans les tests, par exemple conserver `r2 > 0.80` sur l'artefact latest.